# Agente LangGraph + MCP de Playwright: consulta de asientos en SUNARP SPRL

Servicio `SunarpAgentExtractor` (implementa `SunarpPort`): un LLM controla el navegador con `@playwright/mcp`, completa el formulario (oficina, área, partida), resuelve el captcha de imagen con visión, pulsa Buscar y Visualizar, y deja abierta la *Lista de Asientos*.
El array se extrae de forma determinista (regex sobre el snapshot): `[{year, number, original_text}]`.

- **Credenciales:** el LLM nunca las ve; usa la herramienta local `escribir_credencial`.
- **Cloudflare:** el agente solo espera a que el widget se complete en un Chrome real; si falla, termina con `BUSQUEDA_FALLIDA` (no intenta eludirlo).
- **Costos:** el agente tiene prohibido pulsar "Páginas: [n]" (visualizar imágenes/asientos cobra); solo abre la lista.
- **Modelo con visión:** `LLM_MODEL` debe aceptar imágenes (lee el captcha).
- **Requisitos:** Node.js (`npx`), `pip install langgraph langchain langchain-mcp-adapters langchain-openai mcp python-dotenv`, y en `.env`: `LLM_API_KEY`, `LLM_MODEL`, `LLM_PROVIDER`, opcional `LLM_BASE_URL`, `SUNARP_USER`, `SUNARP_PASS`.
- **Perfil persistente:** `.cache/sunarp_mcp_profile` (no ejecutar dos consultas a la vez; cierra otras sesiones MCP que usen el mismo perfil).

| Proveedor | LLM_PROVIDER | LLM_BASE_URL | LLM_MODEL (ej.) | Paquete extra |
|---|---|---|---|---|
| OpenAI | `openai` | *(vacío)* | `gpt-4o-mini` | `langchain-openai` |
| Gemini | `google_genai` | *(vacío)* | `gemini-2.0-flash` | `langchain-google-genai` |
| Anthropic | `anthropic` | *(vacío)* | su modelo | `langchain-anthropic` |

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # raíz del repo
os.chdir(os.path.abspath(".."))            # el perfil .cache/ se crea en la raíz del repo

from app.infrastructure.outbound.extractors.sunarp.agent_extractor import SunarpAgentExtractor

PARAM_OFICINA = "CUSCO"
PARAM_AREA = "Personas Jurídicas"
PARAM_PARTIDA = "11156404"

extractor = SunarpAgentExtractor(max_pasos=60, verbose=True)  # verbose: imprime cada acción del agente
resultado = await extractor.extract(PARAM_OFICINA, PARAM_AREA, PARAM_PARTIDA)

audit = resultado["audit"]
print("\nESTADO:", audit.status.value, "|", audit.duration_ms, "ms")
if audit.details:
    print("DETALLE:", audit.details)
print("ASIENTOS:", [f'{s["year"]}-{s["number"]}' for s in resultado["data"]["seats"]])